# 02 · Retrieval experiments

Embeds the job corpus with `BAAI/bge-small-en-v1.5` and measures each retrieval change with **P@10**: the share of the top 10 jobs whose category label matches the resume's. LLM-judge scores for the same versions are in notebook 03.

| Version | Label P@10 (210 resumes) | Verdict |
|---|---|---|
| Random | 0.106 | |
| Raw resume as query | 0.706 | baseline |
| LLM profile as query | 0.723 | |
| Raw + profile (averaged) | 0.742 | kept |
| + chunked postings | 0.742 | dropped |
| + cross-encoder reranker | 0.741 | dropped |
| + seniority penalty | 0.747 | kept |
| + boilerplate removal | 0.751 | dropped from retrieval |

## Setup and job embeddings

62% of postings are longer than bge-small's 512-token limit, so the end of the description is cut off.

In [ ]:
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

BASE_DIR = r"C:\Users\LENOVO\Documents\Datasets"
OUT_DIR = os.path.join(BASE_DIR, "processed")

corpus = pd.read_pickle(os.path.join(OUT_DIR, "jobs_corpus.pkl"))

model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")
model.max_seq_length = 512
model.half()

In [ ]:
texts = (corpus["title"].fillna("") + ". " + corpus["description"].fillna("")).tolist()

# How much gets cut off at 512 tokens
n_tokens = [len(t) for t in model.tokenizer(texts, add_special_tokens=True)["input_ids"]]
print(f"Postings over 512 tokens (truncated): {np.mean(np.array(n_tokens) > 512):.0%}")

emb = model.encode(texts, batch_size=64, normalize_embeddings=True,
                   show_progress_bar=True, convert_to_numpy=True)

np.save(os.path.join(OUT_DIR, "jobs_emb_bge_small.npy"), emb)
print(f"Embeddings: {emb.shape}  saved to {OUT_DIR}")

## Baseline: raw resume as the query

In [ ]:
eval_set = pd.read_pickle(os.path.join(OUT_DIR, "resumes_eval.pkl"))
corpus_labels = corpus["labels"].apply(set).tolist()

def evaluate(query_emb, categories, name, k=10):
    sims = query_emb @ emb.T
    topk = np.argsort(-sims, axis=1)[:, :k]
    hits = np.array([[cat in corpus_labels[j] for j in row]
                     for row, cat in zip(topk, categories)])
    p_at_k = hits.mean(axis=1)
    per_cat = pd.Series(p_at_k, index=categories).groupby(level=0).mean()
    print(f"[{name}] mean P@{k}: {p_at_k.mean():.3f}")
    return per_cat

# Random baseline: chance a random posting matches each category
label_share = corpus["labels"].explode().value_counts() / len(corpus)
print(f"[random] mean P@10: {label_share.reindex(eval_set['Category']).mean():.3f}")

q_emb = model.encode(eval_set["text"].tolist(), batch_size=32,
                     normalize_embeddings=True, convert_to_numpy=True)
per_cat_raw = evaluate(q_emb, eval_set["Category"].tolist(), "raw resume")

print("\n=== P@10 by category (raw resume)")
print(per_cat_raw.sort_values().round(2).to_string())

[random] mean P@10: 0.080
[raw resume] mean P@10: 0.498

=== P@10 by category (raw resume)
AUTOMOBILE                0.00
ADVOCATE                  0.05
BPO                       0.08
APPAREL                   0.09
HEALTHCARE                0.10
ARTS                      0.11
AVIATION                  0.18
CONSULTANT                0.24
PUBLIC-RELATIONS          0.39
ENGINEERING               0.54
INFORMATION-TECHNOLOGY    0.56
BANKING                   0.61
BUSINESS-DEVELOPMENT      0.64
FITNESS                   0.65
DESIGNER                  0.69
DIGITAL-MEDIA             0.70
CHEF                      0.71
SALES                     0.78
FINANCE                   0.82
CONSTRUCTION              0.84
HR                        0.85
TEACHER                   0.86
ACCOUNTANT                0.97


## Checking the labels

The weakest categories turn out to be label problems, not retrieval failures: the resume dataset's categories often describe the **industry** someone worked in (an "ADVOCATE" resume belongs to a registered nurse), while the job labels describe the **role**. The retrieved jobs below are sensible matches that get scored as wrong.

In [ ]:
sims = q_emb @ emb.T
cats = eval_set["Category"].tolist()

for cat in ["ADVOCATE", "AUTOMOBILE", "HEALTHCARE"]:
    idxs = [i for i, c in enumerate(cats) if c == cat][:2]
    for i in idxs:
        print(f"\n{'='*80}\n[{cat}] resume start:")
        print(eval_set.iloc[i]["text"][:250])
        print("\nTop 5 retrieved:")
        for j in np.argsort(-sims[i])[:5]:
            print(f"  - {corpus.iloc[j]['title'][:60]:<60} {corpus.iloc[j]['labels']}")

[ADVOCATE] resume start:
REGISTERED NURSE CLINICAL CARE COORDINATOR- FIRST AID Professional Background To obtain a position as a registered nurse where my experience and leadership skills can be utilized to provide a positive and effective team environment resulting in excel

Top 5 retrieved:
  - CLINICAL REIMBURSEMENT COORDINATOR-RN                        ['HEALTHCARE']
  - Registered Nurse I, Acute Care - Medical Inpatient Unit      ['HEALTHCARE']
  - RN Care Coordinator (20hrs, Days)                            ['HEALTHCARE']
  - Critical Care Clinical Nurse Coordinator RN                  ['HEALTHCARE']
  - Patient Care Coordinator - Castle Pines Family Practice      ['HEALTHCARE']

[ADVOCATE] resume start:
CLIENT RELATIONS ADVOCATE Career Overview Seeking a challenging and rewarding career in Customer Service that would utilize acquired knowledge, training , and skills to achieve career growth potential and advancement. Dedicated Customer Service Repre

Top 5 retrieved:
  - Client 

In [ ]:
for cat, group in eval_set.groupby("Category"):
    titles = group["text"].str.split().str[:4].str.join(" ").head(5).tolist()
    print(f"\n{cat}:")
    for t in titles:
        print(f"   {t}")

ACCOUNTANT:
   STAFF ACCOUNTANT Summary Skilled
   PAYROLL ACCOUNTANT Summary Sixteen
   ACCOUNTANT Summary To pursue
   ACCOUNTANT Summary If you
   ACCOUNTANT Summary Self-motivated accountant

ADVOCATE:
   REGISTERED NURSE CLINICAL CARE
   CLIENT RELATIONS ADVOCATE Career
   CUSTOMER ADVOCATE Professional Summary
   BENEFIT ADVOCATE Career Overview
   CASE MANAGER/OUTREACH ADVOCATE Summary

APPAREL:
   ASSOCIATE MANAGER OF DESIGN
   CASHIER Summary Responsible Kennel
   RESERVATIONS AGENT/FRONT DESK AGENT/GUEST
   EMBROIDERY MACHINE OPERATOR Summary
   PASSENGER SERVICE AGENT Career

ARTS:
   I.T. SUPPORT TECHNICIAN/SPECIALIST Career
   ADJUNCT FACULTY, ZANVYL KREIGER
   PROJECT COORDINATOR ASSOCIATE Professional
   CREATIVE ASSISTANT Professional Summary
   ADMINISTRATIVE COORDINATOR Summary Reliable

AUTOMOBILE:
   POLICY SERVICE SPECIALIST III
   SERVICES ENGINEER Summary Seeking
   CUSTOMER SERVICE REPRESENTATIVE Core
   STUDENT ATTORNEY - CRIMINAL
   CLINICAL AND DATA OPERATION

## Clean evaluation set

9 categories whose resume titles don't match the label are dropped from evaluation. The decision was made from the titles above, before any score on the reduced set was computed. 15 resumes per remaining category = 210 resumes.

In [ ]:
CLEAN_CATS = ["ACCOUNTANT", "BANKING", "BUSINESS-DEVELOPMENT", "CHEF", "CONSTRUCTION",
              "CONSULTANT", "DESIGNER", "DIGITAL-MEDIA", "ENGINEERING", "FINANCE",
              "HR", "INFORMATION-TECHNOLOGY", "SALES", "TEACHER"]

resumes = pd.read_pickle(os.path.join(OUT_DIR, "resumes_all.pkl"))
eval_clean = (resumes[resumes["Category"].isin(CLEAN_CATS)]
              .groupby("Category").sample(n=15, random_state=42))
eval_clean.to_pickle(os.path.join(OUT_DIR, "resumes_eval_clean.pkl"))
print(f"Clean eval set: {len(eval_clean)} resumes, {eval_clean['Category'].nunique()} categories")

print(f"[random] mean P@10: {label_share.reindex(eval_clean['Category']).mean():.3f}")

q_emb_clean = model.encode(eval_clean["text"].tolist(), batch_size=32,
                           normalize_embeddings=True, convert_to_numpy=True)
per_cat_clean = evaluate(q_emb_clean, eval_clean["Category"].tolist(), "raw resume, clean set")

print("\n=== P@10 by category")
print(per_cat_clean.sort_values().round(2).to_string())

Clean eval set: 210 resumes, 14 categories
[random] mean P@10: 0.106
[raw resume, clean set] mean P@10: 0.706

=== P@10 by category
CONSULTANT                0.28
CONSTRUCTION              0.51
INFORMATION-TECHNOLOGY    0.53
ENGINEERING               0.53
DIGITAL-MEDIA             0.57
BANKING                   0.65
CHEF                      0.75
DESIGNER                  0.77
BUSINESS-DEVELOPMENT      0.79
FINANCE                   0.85
HR                        0.88
SALES                     0.90
TEACHER                   0.91
ACCOUNTANT                0.95


## LLM profile as the query

Each resume is condensed by an LLM into a short profile (notebook 03). Profiles help technical resumes, whose skills were cut off at 512 tokens, and hurt people-facing ones, whose raw wording already matched job postings. Averaging the two embeddings keeps most of both.

In [ ]:
eval_clean = pd.read_pickle(os.path.join(OUT_DIR, "resumes_eval_clean_profiles.pkl"))
cats = eval_clean["Category"].tolist()
corpus_labels = corpus["labels"].apply(set).tolist()

def evaluate(query_emb, categories, name, k=10):
    sims = query_emb @ emb.T
    topk = np.argsort(-sims, axis=1)[:, :k]
    hits = np.array([[cat in corpus_labels[j] for j in row]
                     for row, cat in zip(topk, categories)])
    p_at_k = hits.mean(axis=1)
    print(f"[{name}] mean P@{k}: {p_at_k.mean():.3f}")
    return pd.Series(p_at_k, index=categories).groupby(level=0).mean()

def profile_to_text(p):
    return (f"{p['title']}. Seniority: {p['seniority']}. {p['summary']} "
            f"Skills: {', '.join(p['skills'])}. Domains: {', '.join(p['domains'])}.")

def encode(texts):
    return model.encode(texts, batch_size=32, normalize_embeddings=True, convert_to_numpy=True)

q_raw = encode(eval_clean["text"].tolist())
q_prof = encode([profile_to_text(p) for p in eval_clean["profile"]])

per_cat = pd.DataFrame({
    "raw": evaluate(q_raw, cats, "raw resume"),
    "profile": evaluate(q_prof, cats, "LLM profile"),
})
per_cat["change"] = per_cat["profile"] - per_cat["raw"]
print("\n=== P@10 by category")
print(per_cat.sort_values("change").round(2).to_string())

[raw resume] mean P@10: 0.706
[LLM profile] mean P@10: 0.723

=== P@10 by category
                         raw  profile  change
SALES                   0.90     0.79   -0.11
CHEF                    0.75     0.69   -0.06
TEACHER                 0.91     0.87   -0.04
BUSINESS-DEVELOPMENT    0.79     0.77   -0.03
BANKING                 0.65     0.65   -0.01
FINANCE                 0.85     0.85   -0.00
ACCOUNTANT              0.95     0.97    0.02
DESIGNER                0.77     0.79    0.02
HR                      0.88     0.91    0.03
CONSTRUCTION            0.51     0.55    0.04
DIGITAL-MEDIA           0.57     0.63    0.05
CONSULTANT              0.28     0.37    0.09
INFORMATION-TECHNOLOGY  0.53     0.63    0.09
ENGINEERING             0.53     0.67    0.13


In [ ]:
q_combo = q_raw + q_prof
q_combo = q_combo / np.linalg.norm(q_combo, axis=1, keepdims=True)

per_cat["combo"] = evaluate(q_combo, cats, "raw + profile")
print("\n=== P@10 by category")
print(per_cat[["raw", "profile", "combo"]].sort_values("combo").round(2).to_string())

[raw + profile] mean P@10: 0.742

=== P@10 by category
                         raw  profile  combo
CONSULTANT              0.28     0.37   0.37
CONSTRUCTION            0.51     0.55   0.53
ENGINEERING             0.53     0.67   0.59
INFORMATION-TECHNOLOGY  0.53     0.63   0.64
DIGITAL-MEDIA           0.57     0.63   0.65
BANKING                 0.65     0.65   0.71
CHEF                    0.75     0.69   0.72
DESIGNER                0.77     0.79   0.79
BUSINESS-DEVELOPMENT    0.79     0.77   0.82
SALES                   0.90     0.79   0.87
TEACHER                 0.91     0.87   0.89
FINANCE                 0.85     0.85   0.89
HR                      0.88     0.91   0.94
ACCOUNTANT              0.95     0.97   0.98


## Chunked job postings (dropped)

Scoring each job by its best-matching 250-word chunk replaced about half of each top 10 but left P@10 unchanged. The LLM judge (notebook 03) rated it slightly worse, so whole postings are kept. This cell also exports the candidate lists for the judge.

In [ ]:
def chunk_words(text, size=250, overlap=50):
    words = text.split()
    if len(words) <= size:
        return [" ".join(words)]
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, len(words) - overlap, step)]

chunk_texts, chunk_job = [], []
for j, (title, desc) in enumerate(zip(corpus["title"].fillna(""), corpus["description"].fillna(""))):
    for c in chunk_words(desc):
        chunk_texts.append(f"{title}. {c}")
        chunk_job.append(j)
chunk_job = np.array(chunk_job)
print(f"{len(chunk_texts):,} chunks from {len(corpus):,} postings "
      f"(avg {len(chunk_texts) / len(corpus):.1f} per posting)")

emb_chunks = model.encode(chunk_texts, batch_size=64, normalize_embeddings=True,
                          show_progress_bar=True, convert_to_numpy=True)
np.save(os.path.join(OUT_DIR, "jobs_chunk_emb_bge_small.npy"), emb_chunks)
np.save(os.path.join(OUT_DIR, "jobs_chunk_job_idx.npy"), chunk_job)

def evaluate_scores(scores, categories, name, k=10):
    topk = np.argsort(-scores, axis=1)[:, :k]
    hits = np.array([[cat in corpus_labels[j] for j in row]
                     for row, cat in zip(topk, categories)])
    p_at_k = hits.mean(axis=1)
    print(f"[{name}] mean P@{k}: {p_at_k.mean():.3f}")
    return pd.Series(p_at_k, index=categories).groupby(level=0).mean()

# Job score = best chunk score (chunks are stored in job order, so reduceat works)
starts = np.flatnonzero(np.r_[True, chunk_job[1:] != chunk_job[:-1]])
job_scores_chunked = np.maximum.reduceat(q_combo @ emb_chunks.T, starts, axis=1)

per_cat["combo_chunked"] = evaluate_scores(job_scores_chunked, cats, "raw + profile, chunked jobs")
print("\n=== P@10 by category")
print(per_cat[["combo", "combo_chunked"]].sort_values("combo_chunked").round(2).to_string())

48,398 chunks from 17,145 postings (avg 2.8 per posting)
[raw + profile, chunked jobs] mean P@10: 0.742

=== P@10 by category
                        combo  combo_chunked
CONSULTANT               0.37           0.33
CONSTRUCTION             0.53           0.52
ENGINEERING              0.59           0.63
INFORMATION-TECHNOLOGY   0.64           0.68
BANKING                  0.71           0.69
DIGITAL-MEDIA            0.65           0.71
CHEF                     0.72           0.73
BUSINESS-DEVELOPMENT     0.82           0.80
DESIGNER                 0.79           0.80
SALES                    0.87           0.82
TEACHER                  0.89           0.87
FINANCE                  0.89           0.89
HR                       0.94           0.96
ACCOUNTANT               0.98           0.97


In [ ]:
top_whole = np.argsort(-(q_combo @ emb.T), axis=1)[:, :10]
top_chunked = np.argsort(-job_scores_chunked, axis=1)[:, :10]

overlap = np.mean([len(set(a) & set(b)) for a, b in zip(top_whole, top_chunked)])
print(f"Avg overlap between the two top-10 lists: {overlap:.1f} / 10")

sample_pos = (eval_clean.reset_index(drop=True)
              .groupby("Category").sample(n=3, random_state=0).index.to_numpy())

rows = []
for i in sample_pos:
    r = eval_clean.iloc[i]
    rows.append({
        "ID": r["ID"], "Category": r["Category"], "profile": r["profile"],
        "top_whole": [int(j) for j in top_whole[i]],
        "top_chunked": [int(j) for j in top_chunked[i]],
        "candidates": sorted({int(j) for j in top_whole[i]} | {int(j) for j in top_chunked[i]}),
    })
judge_df = pd.DataFrame(rows)

n_pairs = judge_df["candidates"].str.len().sum()
print(f"{len(judge_df)} resumes, {n_pairs} unique resume-job pairs to judge "
      f"(avg {n_pairs / len(judge_df):.1f} per resume)")

judge_df.to_pickle(os.path.join(OUT_DIR, "judge_candidates.pkl"))
print("Saved judge_candidates.pkl")

Avg overlap between the two top-10 lists: 5.1 / 10
42 resumes, 639 unique resume-job pairs to judge (avg 15.2 per resume)
Saved judge_candidates.pkl


## Cross-encoder reranker (dropped)

`BAAI/bge-reranker-base` re-scoring the top 50: P@10 flat, and the LLM judge rated it worse on 18 resumes and better on 9. It is trained for search relevance (topic overlap), not job fit.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-base", max_length=512, device="cuda")
reranker.model.half()

TOP_N = 50
scores_combo = q_combo @ emb.T
top50 = np.argsort(-scores_combo, axis=1)[:, :TOP_N]

queries = [profile_to_text(p) for p in eval_clean["profile"]]
job_texts = (corpus["title"].fillna("") + ". " + corpus["description"].fillna("")).tolist()

rr_pairs = [(queries[i], job_texts[j]) for i in range(len(queries)) for j in top50[i]]
ce = reranker.predict(rr_pairs, batch_size=32, show_progress_bar=True).reshape(len(queries), TOP_N)

# Put reranker scores back into a full score matrix so evaluate_scores works unchanged
rerank_scores = np.full_like(scores_combo, -np.inf)
np.put_along_axis(rerank_scores, top50, ce, axis=1)
per_cat["rerank"] = evaluate_scores(rerank_scores, cats, "raw + profile -> rerank top 50")

top10_rerank = np.argsort(-rerank_scores, axis=1)[:, :10]
rerank_out = {int(i): [int(j) for j in row] for i, row in zip(eval_clean["ID"], top10_rerank)}
pd.to_pickle(rerank_out, os.path.join(OUT_DIR, "rerank_top10.pkl"))
print("Saved rerank_top10.pkl")

print("\n=== P@10 by category")
print(per_cat[["combo", "rerank"]].sort_values("rerank").round(2).to_string())

[raw + profile -> rerank top 50] mean P@10: 0.741
Saved rerank_top10.pkl

=== P@10 by category
                        combo  rerank
CONSULTANT               0.37    0.41
CONSTRUCTION             0.53    0.54
DIGITAL-MEDIA            0.65    0.57
BANKING                  0.71    0.59
INFORMATION-TECHNOLOGY   0.64    0.63
ENGINEERING              0.59    0.66
CHEF                     0.72    0.74
DESIGNER                 0.79    0.83
SALES                    0.87    0.85
FINANCE                  0.89    0.87
TEACHER                  0.89    0.88
HR                       0.94    0.89
BUSINESS-DEVELOPMENT     0.82    0.95
ACCOUNTANT               0.98    0.97


## Seniority penalty (kept)

Jobs whose experience level doesn't suit the candidate's seniority are demoted by a fixed penalty. The penalty size is set from the score distribution (the median rank-10 to rank-50 gap) before evaluating, not tuned on results.

In [ ]:
print("=== Job experience levels (corpus)")
print(corpus["formatted_experience_level"].value_counts(dropna=False).to_string())

print("\n=== Profile seniority (eval resumes)")
print(eval_clean["profile"].apply(lambda p: p["seniority"]).value_counts().to_string())

=== Job experience levels (corpus)
formatted_experience_level
Mid-Senior level    6000
NaN                 4126
Entry level         3935
Associate           1857
Director             712
Internship           299
Executive            216

=== Profile seniority (eval resumes)
profile
senior       114
mid           43
executive     41
entry         12


In [ ]:
LEVEL_OK = {
    "entry":     {"Internship", "Entry level", "Associate"},
    "mid":       {"Entry level", "Associate", "Mid-Senior level"},
    "senior":    {"Associate", "Mid-Senior level", "Director"},
    "executive": {"Mid-Senior level", "Director", "Executive"},
}
levels = corpus["formatted_experience_level"]
job_bad = {s: (levels.notna() & ~levels.isin(ok)).to_numpy() for s, ok in LEVEL_OK.items()}
prof_sen = eval_clean["profile"].apply(lambda p: p["seniority"]).tolist()
mismatch = np.stack([job_bad[s] for s in prof_sen]).astype(np.float32)

# Penalty size from the score distribution, fixed before evaluating
srt = -np.sort(-scores_combo, axis=1)
LAMBDA = float(np.median(srt[:, 9] - srt[:, 49]))
print(f"Median gap rank1->rank10: {np.median(srt[:, 0] - srt[:, 9]):.3f} | "
      f"rank10->rank50: {LAMBDA:.3f}  ->  penalty = {LAMBDA:.3f}")

top10_combo = np.argsort(-scores_combo, axis=1)[:, :10]
was_bad = np.take_along_axis(mismatch, top10_combo, axis=1).mean()
print(f"Share of current top-10 jobs with mismatched seniority: {was_bad:.0%}")

scores_sen = scores_combo - LAMBDA * mismatch
per_cat["seniority"] = evaluate_scores(scores_sen, cats, "raw + profile + seniority")

top10_sen = np.argsort(-scores_sen, axis=1)[:, :10]
overlap = np.mean([len(set(a) & set(b)) for a, b in zip(top10_combo, top10_sen)])
print(f"Avg overlap with current top-10: {overlap:.1f} / 10")

sen_out = {int(i): [int(j) for j in row] for i, row in zip(eval_clean["ID"], top10_sen)}
pd.to_pickle(sen_out, os.path.join(OUT_DIR, "seniority_top10.pkl"))
print("Saved seniority_top10.pkl")

Median gap rank1->rank10: 0.033 | rank10->rank50: 0.028  ->  penalty = 0.028
Share of current top-10 jobs with mismatched seniority: 11%
[raw + profile + seniority] mean P@10: 0.747
Avg overlap with current top-10: 9.0 / 10
Saved seniority_top10.pkl


## Boilerplate removal (dropped from retrieval)

A sentence counts as boilerplate if it appears under 3 or more different job titles (company intros, equal-opportunity text, benefits). No measurable retrieval gain, but the cleaned descriptions are used as LLM input in notebook 03.

In [ ]:
import re
from collections import defaultdict

def sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", str(text)) if len(s.split()) >= 5]

sent_lists = corpus["description"].apply(sentences)

titles_per_sent = defaultdict(set)
for title, sl in zip(corpus["title"].str.lower(), sent_lists):
    for s in set(sl):
        titles_per_sent[s].add(title)
n_titles = {s: len(t) for s, t in titles_per_sent.items()}

MIN_TITLES = 3
print("=== Most widespread sentences (number of distinct job titles)")
for s, c in sorted(n_titles.items(), key=lambda x: -x[1])[:12]:
    print(f"{c:5d}  {s[:110]}")

def boiler_share(sl):
    words = sum(len(s.split()) for s in sl)
    bad = sum(len(s.split()) for s in sl if n_titles[s] >= MIN_TITLES)
    return bad / words if words else 0

share = sent_lists.apply(boiler_share)
print(f"\nAvg share of words flagged as boilerplate: {share.mean():.0%}")
print(f"Postings with >50% flagged: {(share > 0.5).mean():.0%}")
print(f"Postings with nothing flagged: {(share == 0).mean():.0%}")

ex = share[(share > 0.3) & (share < 0.6)].index[0]
print(f"\n=== Example posting: {corpus.loc[ex, 'title']}")
for s in sent_lists[ex]:
    tag = "REMOVE" if n_titles[s] >= MIN_TITLES else "keep  "
    print(f"[{tag}] {s[:120]}")

=== Most widespread sentences (number of distinct job titles)
  241  Reasonable accommodations may be made to enable individuals with disabilities to perform the essential functio
   88  Amazon is an equal opportunity employer and does not discriminate on the basis of race, national origin, gende
   87  For individuals with disabilities who would like to request an accommodation, please visit https://www.amazon.
   77  Performs other duties as assigned.
   77  There is emphasis on proactive monitoring, governance, risk identification and escalation, as well as making s
   77  They are accountable for execution of all applicable risk programs (Credit, Market, Financial Crimes, Operatio
   77  Please see our Drug and Alcohol Policy to learn more.
   77  Employees support our focus on building strong customer relationships balanced with a strong risk mitigating a
   77  Candidates applying to job openings posted in US: All qualified applicants will receive consideration for empl
   77  Ap

In [ ]:
def clean_desc(text):
    parts = [s.strip() for s in re.split(r"(?<=[.!?])\s+", str(text))]
    kept = [s for s in parts if n_titles.get(s, 0) < MIN_TITLES]
    cleaned = " ".join(kept)
    return cleaned if len(cleaned.split()) >= 30 else str(text)

corpus["description_clean"] = corpus["description"].apply(clean_desc)
fell_back = (corpus["description_clean"] == corpus["description"].astype(str)) & (share > 0)
print(f"Postings kept original (too little left after cleaning): {fell_back.sum()}")

texts_clean = (corpus["title"].fillna("") + ". " + corpus["description_clean"]).tolist()
n_tok = [len(t) for t in model.tokenizer(texts_clean)["input_ids"]]
print(f"Postings over 512 tokens: before 62% -> after {np.mean(np.array(n_tok) > 512):.0%}")

emb_clean = model.encode(texts_clean, batch_size=64, normalize_embeddings=True,
                         show_progress_bar=True, convert_to_numpy=True)
np.save(os.path.join(OUT_DIR, "jobs_emb_bge_small_clean.npy"), emb_clean)
corpus.to_pickle(os.path.join(OUT_DIR, "jobs_corpus_clean.pkl"))

scores_clean = q_combo @ emb_clean.T - LAMBDA * mismatch
per_cat["clean"] = evaluate_scores(scores_clean, cats, "raw + profile + seniority, clean jobs")

top10_clean = np.argsort(-scores_clean, axis=1)[:, :10]
overlap = np.mean([len(set(a) & set(b)) for a, b in zip(top10_sen, top10_clean)])
print(f"Avg overlap with current best top-10: {overlap:.1f} / 10")

clean_out = {int(i): [int(j) for j in row] for i, row in zip(eval_clean["ID"], top10_clean)}
pd.to_pickle(clean_out, os.path.join(OUT_DIR, "clean_top10.pkl"))
print("Saved clean_top10.pkl")

Postings kept original (too little left after cleaning): 83
Postings over 512 tokens: before 62% -> after 52%
[raw + profile + seniority, clean jobs] mean P@10: 0.751
Avg overlap with current best top-10: 8.9 / 10
Saved clean_top10.pkl


## Export top-20 candidates for LLM ranking

In [ ]:
top20_sen = np.argsort(-scores_sen, axis=1)[:, :20]
top20_out = {int(i): [int(j) for j in row] for i, row in zip(eval_clean["ID"], top20_sen)}
pd.to_pickle(top20_out, os.path.join(OUT_DIR, "seniority_top20.pkl"))
print(f"Saved top-20 candidates for {len(top20_out)} resumes")

Saved top-20 candidates for 210 resumes
